# ✈️ Avionics SRD Generator (Hugging Face Inference API + Gradio)
No local model download. Uses Hugging Face hosted inference.


In [ ]:
!pip install gradio requests python-docx reportlab

In [ ]:
import gradio as gr
import requests
from docx import Document
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet


In [ ]:
# Add your Hugging Face API token
HF_TOKEN = "YOUR_HF_TOKEN"
API_URL = "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

In [ ]:
def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()


In [ ]:
def generate_srd(system_name, complexity):
    prompt = f"Generate a detailed avionics SRD for {system_name} with {complexity} complexity including faults, frames, interfaces, logging."
    output = query({"inputs": prompt})
    try:
        return output[0]['generated_text']
    except:
        return str(output)


In [ ]:
def export_word(text):
    doc = Document()
    doc.add_heading('SRD Document', 0)
    doc.add_paragraph(text)
    path = 'srd.docx'
    doc.save(path)
    return path

In [ ]:
def export_pdf(text):
    styles = getSampleStyleSheet()
    pdf = SimpleDocTemplate('srd.pdf')
    story = [Paragraph(text, styles['Normal'])]
    pdf.build(story)
    return 'srd.pdf'

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(primary_hue='blue')) as app:
    gr.Markdown('# ✈️ Avionics SRD Generator (HF API)')

    system = gr.Textbox(label='System Name')
    complexity = gr.Dropdown(['Low','Medium','High'], value='High')

    generate_btn = gr.Button('Generate SRD')
    output = gr.Textbox(lines=20)

    word_btn = gr.Button('Download Word')
    pdf_btn = gr.Button('Download PDF')
    file_out = gr.File()

    generate_btn.click(generate_srd, [system, complexity], output)
    word_btn.click(export_word, output, file_out)
    pdf_btn.click(export_pdf, output, file_out)

app.launch()